# Manual Annotation Result Evaluation for Lactate, Shock and Coma

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
)

In [6]:
# =========================================================
# paths
# =========================================================
COMA_PATH = "coma_manual_annotation_sample - coma_manual_annotation_sample.csv"
SHOCK_PATH = "shock_annotation_comp - shock_manual_annotation_sample.csv"
LACTATE_PATH = "lactate_annotation_comp - lactate_validation_set.csv"


# =========================================================
# helpers
# =========================================================
def normalize_bool(x):
    """
    Converts boolean-like values to:
    1 = True
    0 = False
    np.nan = missing/unknown
    """
    if pd.isna(x):
        return np.nan

    if isinstance(x, bool):
        return int(x)

    s = str(x).strip().lower()

    if s in {"true", "1", "yes", "y"}:
        return 1
    if s in {"false", "0", "no", "n"}:
        return 0
    if s in {"null", "none", ""}:
        return np.nan

    return np.nan


def normalize_text_or_null(x):
    """
    Normalize text fields such as severity.
    Returns lowercased string or np.nan for null-like values.
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    if s in {"", "null", "none", "nan"}:
        return np.nan
    return s


def normalize_float_or_null(x):
    """
    Normalize numeric field.
    Returns float or np.nan.
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    if s in {"", "null", "none", "nan"}:
        return np.nan

    try:
        return float(s)
    except:
        return np.nan


def values_match_with_null(a, b):
    """
    Match rule:
    - both null => match
    - one null and one not => mismatch
    - both non-null => exact equality
    """
    if pd.isna(a) and pd.isna(b):
        return True
    if pd.isna(a) or pd.isna(b):
        return False
    return a == b


def print_binary_metrics(y_true, y_pred, title="Binary evaluation"):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=[0, 1], zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"\n{'=' * 70}")
    print(title)
    print(f"{'=' * 70}")
    print(f"Accuracy: {acc:.4f}")
    print("\nConfusion matrix [rows=true, cols=pred] with labels [0,1]:")
    print(cm)

    print("\nPer-class metrics:")
    print(f"Class 0 -> Precision: {precision[0]:.4f} | Recall: {recall[0]:.4f} | F1: {f1[0]:.4f} | Support: {support[0]}")
    print(f"Class 1 -> Precision: {precision[1]:.4f} | Recall: {recall[1]:.4f} | F1: {f1[1]:.4f} | Support: {support[1]}")

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))


# =========================================================
# 1) COMA EVALUATION
# =========================================================
def evaluate_coma(path=COMA_PATH):
    df = pd.read_csv(path)

    y_true = df["manual_coma_present"].map(normalize_bool)
    y_pred = df["llm_present"].map(normalize_bool)

    mask = y_true.notna() & y_pred.notna()
    df_eval = df.loc[mask].copy()

    y_true = y_true.loc[mask].astype(int)
    y_pred = y_pred.loc[mask].astype(int)

    print_binary_metrics(y_true, y_pred, title="COMA / UNRESPONSIVENESS EVALUATION")

    mismatches = df_eval.loc[y_true.values != y_pred.values, [
        "note_id", "hadm_id", "subject_id", "llm_present", "manual_coma_present",
        "llm_evidence_quote", "reason"
    ]]

    print("\nNumber of mismatches:", len(mismatches))
    #if len(mismatches) > 0:
        #print(mismatches.to_string(index=False))

    return df_eval, mismatches


# =========================================================
# 2) SHOCK EVALUATION
# =========================================================
def evaluate_shock(path=SHOCK_PATH):
    df = pd.read_csv(path)

    # ---------------------------
    # binary presence evaluation
    # ---------------------------
    y_true = df["manual_shock_present"].map(normalize_bool)
    y_pred = df["llm_present"].map(normalize_bool)

    mask = y_true.notna() & y_pred.notna()
    df_eval = df.loc[mask].copy()

    y_true_bin = y_true.loc[mask].astype(int)
    y_pred_bin = y_pred.loc[mask].astype(int)

    print_binary_metrics(y_true_bin, y_pred_bin, title="SHOCK PRESENCE EVALUATION")

    binary_mismatches = df_eval.loc[y_true_bin.values != y_pred_bin.values, [
        "note_id", "hadm_id", "subject_id", "llm_present", "manual_shock_present",
        "llm_severity", "manual_shock_severity", "llm_evidence_quote", "reason"
    ]]

    print("\nNumber of binary mismatches:", len(binary_mismatches))
    #if len(binary_mismatches) > 0:
        #print(binary_mismatches.to_string(index=False))

    # ---------------------------
    # severity evaluation
    # includes null/null as MATCH
    # ---------------------------
    df_sev = df.copy()
    df_sev["manual_severity_norm"] = df_sev["manual_shock_severity"].map(normalize_text_or_null)
    df_sev["llm_severity_norm"] = df_sev["llm_severity"].map(normalize_text_or_null)

    df_sev["severity_match"] = df_sev.apply(
        lambda row: values_match_with_null(row["manual_severity_norm"], row["llm_severity_norm"]),
        axis=1
    )

    severity_accuracy_including_nulls = df_sev["severity_match"].mean()

    print(f"\n{'=' * 70}")
    print("SHOCK SEVERITY EXACT MATCH EVALUATION (INCLUDING NULL/NULL MATCHES)")
    print(f"{'=' * 70}")
    print(f"Total rows compared: {len(df_sev)}")
    print(f"Exact match accuracy: {severity_accuracy_including_nulls:.4f}")

    sev_match_counts = df_sev["severity_match"].value_counts(dropna=False)
    print("\nMatch counts:")
    print(sev_match_counts)

    sev_mismatches = df_sev.loc[~df_sev["severity_match"], [
        "note_id", "hadm_id", "subject_id",
        "llm_present", "manual_shock_present",
        "llm_severity", "manual_shock_severity",
        "llm_evidence_quote", "reason"
    ]]

    print("\nNumber of severity mismatches (including null logic):", len(sev_mismatches))
    #if len(sev_mismatches) > 0:
    #    print(sev_mismatches.to_string(index=False))

    # optional: confusion matrix only on rows where both are non-null
    severity_order = {"none": 0, "mild": 1, "moderate": 2, "severe": 3}
    nonnull_mask = df_sev["manual_severity_norm"].notna() & df_sev["llm_severity_norm"].notna()

    if nonnull_mask.sum() > 0:
        y_true_sev = df_sev.loc[nonnull_mask, "manual_severity_norm"].map(severity_order)
        y_pred_sev = df_sev.loc[nonnull_mask, "llm_severity_norm"].map(severity_order)

        print(f"\n{'=' * 70}")
        print("SHOCK SEVERITY CLASSIFICATION REPORT (ONLY NON-NULL PAIRS)")
        print(f"{'=' * 70}")
        print(f"Rows with both sides non-null: {nonnull_mask.sum()}")
        print(confusion_matrix(y_true_sev, y_pred_sev, labels=[0, 1, 2, 3]))
        print(classification_report(y_true_sev, y_pred_sev, labels=[0, 1, 2, 3], digits=4, zero_division=0))

    return df_eval, binary_mismatches, df_sev, sev_mismatches


# =========================================================
# 3) LACTATE EVALUATION
# =========================================================
def evaluate_lactate(path=LACTATE_PATH):
    df = pd.read_csv(path)

    # ---------------------------
    # binary present / absent / null exact-match evaluation
    # includes null/null as MATCH
    # ---------------------------
    df_pres = df.copy()
    df_pres["human_present_norm"] = df_pres["human_present"].map(normalize_bool)
    df_pres["llm_present_norm"] = df_pres["llm_present"].map(normalize_bool)

    df_pres["presence_match"] = df_pres.apply(
        lambda row: values_match_with_null(row["human_present_norm"], row["llm_present_norm"]),
        axis=1
    )

    presence_accuracy_including_nulls = df_pres["presence_match"].mean()

    print(f"\n{'=' * 70}")
    print("LACTATE PRESENCE EXACT MATCH EVALUATION (INCLUDING NULL/NULL MATCHES)")
    print(f"{'=' * 70}")
    print(f"Total rows compared: {len(df_pres)}")
    print(f"Exact match accuracy: {presence_accuracy_including_nulls:.4f}")

    presence_match_counts = df_pres["presence_match"].value_counts(dropna=False)
    print("\nMatch counts:")
    print(presence_match_counts)

    presence_mismatches = df_pres.loc[~df_pres["presence_match"], [
        "note_id", "hadm_id", "subject_id",
        "llm_present", "human_present",
        "llm_lactate_value", "human_lactate_value",
        "llm_evidence_quote"
    ]]

    print("\nNumber of lactate presence mismatches (including null logic):", len(presence_mismatches))
    #if len(presence_mismatches) > 0:
        #print(presence_mismatches.to_string(index=False))

    # optional: standard binary classification report only on non-null pairs
    nonnull_presence_mask = (
        df_pres["human_present_norm"].notna() & df_pres["llm_present_norm"].notna()
    )

    if nonnull_presence_mask.sum() > 0:
        y_true_bin = df_pres.loc[nonnull_presence_mask, "human_present_norm"].astype(int)
        y_pred_bin = df_pres.loc[nonnull_presence_mask, "llm_present_norm"].astype(int)

        print_binary_metrics(
            y_true_bin,
            y_pred_bin,
            title="LACTATE PRESENCE CLASSIFICATION REPORT (ONLY NON-NULL PAIRS)"
        )

    # ---------------------------
    # numeric value exact-match evaluation
    # includes null/null as MATCH
    # ---------------------------
    df_val = df.copy()
    df_val["human_lactate_value_norm"] = df_val["human_lactate_value"].map(normalize_float_or_null)
    df_val["llm_lactate_value_norm"] = df_val["llm_lactate_value"].map(normalize_float_or_null)

    df_val["value_match"] = df_val.apply(
        lambda row: values_match_with_null(row["human_lactate_value_norm"], row["llm_lactate_value_norm"]),
        axis=1
    )

    value_accuracy_including_nulls = df_val["value_match"].mean()

    print(f"\n{'=' * 70}")
    print("LACTATE NUMERIC EXACT MATCH EVALUATION (INCLUDING NULL/NULL MATCHES)")
    print(f"{'=' * 70}")
    print(f"Total rows compared: {len(df_val)}")
    print(f"Exact match accuracy: {value_accuracy_including_nulls:.4f}")

    value_match_counts = df_val["value_match"].value_counts(dropna=False)
    print("\nMatch counts:")
    print(value_match_counts)

    value_mismatches = df_val.loc[~df_val["value_match"], [
        "note_id", "hadm_id", "subject_id",
        "llm_lactate_value", "human_lactate_value",
        "llm_evidence_quote"
    ]]

    print("\nNumber of lactate value mismatches (including null logic):", len(value_mismatches))
    #if len(value_mismatches) > 0:
        #print(value_mismatches.to_string(index=False))

    # optional numeric agreement metrics only on rows where both are non-null
    nonnull_value_mask = (
        df_val["human_lactate_value_norm"].notna() &
        df_val["llm_lactate_value_norm"].notna()
    )

    if nonnull_value_mask.sum() > 0:
        y_true_val = df_val.loc[nonnull_value_mask, "human_lactate_value_norm"].astype(float)
        y_pred_val = df_val.loc[nonnull_value_mask, "llm_lactate_value_norm"].astype(float)

        mae = mean_absolute_error(y_true_val, y_pred_val)
        rmse = np.sqrt(np.mean((y_true_val - y_pred_val) ** 2))
        corr = np.corrcoef(y_true_val, y_pred_val)[0, 1] if len(y_true_val) > 1 else np.nan

        print(f"\n{'=' * 70}")
        print("LACTATE NUMERIC AGREEMENT METRICS (ONLY NON-NULL PAIRS)")
        print(f"{'=' * 70}")
        print(f"N compared: {len(y_true_val)}")
        print(f"MAE:  {mae:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"Correlation: {corr:.4f}")

    return df_pres, presence_mismatches, df_val, value_mismatches


# =========================================================
# run all
# =========================================================
if __name__ == "__main__":
    evaluate_coma()
    evaluate_shock()
    evaluate_lactate()


COMA / UNRESPONSIVENESS EVALUATION
Accuracy: 0.9833

Confusion matrix [rows=true, cols=pred] with labels [0,1]:
[[29  0]
 [ 1 30]]

Per-class metrics:
Class 0 -> Precision: 0.9667 | Recall: 1.0000 | F1: 0.9831 | Support: 29
Class 1 -> Precision: 1.0000 | Recall: 0.9677 | F1: 0.9836 | Support: 31

Classification report:
              precision    recall  f1-score   support

           0     0.9667    1.0000    0.9831        29
           1     1.0000    0.9677    0.9836        31

    accuracy                         0.9833        60
   macro avg     0.9833    0.9839    0.9833        60
weighted avg     0.9839    0.9833    0.9833        60


Number of mismatches: 1

SHOCK PRESENCE EVALUATION
Accuracy: 0.8077

Confusion matrix [rows=true, cols=pred] with labels [0,1]:
[[12  2]
 [ 8 30]]

Per-class metrics:
Class 0 -> Precision: 0.6000 | Recall: 0.8571 | F1: 0.7059 | Support: 14
Class 1 -> Precision: 0.9375 | Recall: 0.7895 | F1: 0.8571 | Support: 38

Classification report:
             